In [1]:
import pandas as pd
import sys
from pathlib import Path

In [2]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

In [3]:
from src.load_data import load_data

In [4]:
df = load_data()

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X = df.drop("y", axis=1)
y = df["y"].map({"no": 0, "yes": 1})

In [7]:
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [8]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

In [9]:
from sklearn.dummy import DummyClassifier

In [10]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(x_train, y_train)
y_pred = dummy_model.predict(x_test)

In [11]:
df["y"].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [16]:
baseline_results ={
    "Model": "Dummy Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, pos_label=1),
    "Recall": recall_score(y_test, y_pred, pos_label=1),
    "F1 Score": f1_score(y_test, y_pred, pos_label=1),
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
}
baseline_results_df = pd.DataFrame([baseline_results])
print(baseline_results_df)

              Model  Accuracy  Precision  Recall  F1 Score  \
0  Dummy Classifier  0.879354        0.0     0.0       0.0   

         confusion_matrix  
0  [[7952, 0], [1091, 0]]  


c:\Users\deniz\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [17]:
baseline_results_df.to_csv(ROOT_DIR / "reports" / "baseline_results.csv", index=False)

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [19]:
categorical_features = ["job", "marital", "education", "contact"]
numerical_features = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

In [20]:
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [21]:
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

In [22]:
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numerical_pipeline, numerical_features)
])

In [23]:
x_train_processed = preprocessor.fit_transform(x_train)
x_val_processed = preprocessor.fit_transform(x_val)

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [25]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
    "CatBoost": CatBoostClassifier()
}

In [26]:
result = []

In [27]:
for model_name, model in models.items():
    model.fit(x_train_processed, y_train)
    y_pred = model.predict(x_val_processed)

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_val, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_val, y_pred, pos_label=1, zero_division=0)
    conf_matrix = confusion_matrix(y_val, y_pred, labels=[0, 1]).tolist()

    result.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

Learning rate set to 0.043347
0:	learn: 0.6453735	total: 154ms	remaining: 2m 34s
1:	learn: 0.6067521	total: 164ms	remaining: 1m 21s
2:	learn: 0.5678242	total: 174ms	remaining: 57.9s
3:	learn: 0.5275584	total: 184ms	remaining: 45.7s
4:	learn: 0.5033278	total: 193ms	remaining: 38.4s
5:	learn: 0.4743730	total: 202ms	remaining: 33.5s
6:	learn: 0.4479063	total: 213ms	remaining: 30.3s
7:	learn: 0.4244504	total: 226ms	remaining: 28.1s
8:	learn: 0.4064851	total: 236ms	remaining: 26s
9:	learn: 0.3918245	total: 246ms	remaining: 24.3s
10:	learn: 0.3765739	total: 255ms	remaining: 22.9s
11:	learn: 0.3665697	total: 264ms	remaining: 21.7s
12:	learn: 0.3554305	total: 274ms	remaining: 20.8s
13:	learn: 0.3476042	total: 284ms	remaining: 20s
14:	learn: 0.3365901	total: 294ms	remaining: 19.3s
15:	learn: 0.3276869	total: 303ms	remaining: 18.6s
16:	learn: 0.3206956	total: 312ms	remaining: 18s
17:	learn: 0.3137558	total: 321ms	remaining: 17.5s
18:	learn: 0.3097822	total: 330ms	remaining: 17.1s
19:	learn: 0.30

In [28]:
results_df = pd.DataFrame(result)
results_df.to_csv(ROOT_DIR / "reports" / "model_results.csv", index=False)

In [29]:
from sklearn.model_selection import RandomizedSearchCV

In [30]:
dt_params = {
    "max_depth": [3,5,7,10,15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy", "log_loss"],
    "class_weight": [None, "balanced"]
}

In [31]:
dt_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=dt_params,
    n_iter=30,
    scoring = "f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [42]:
dt_search.fit(x_train_processed, y_train)

print("Best parameters found: ", dt_search.best_params_)
print("Best CV F1-score: ", dt_search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found:  {'min_samples_split': 20, 'min_samples_leaf': 10, 'max_depth': 10, 'criterion': 'entropy', 'class_weight': 'balanced'}
Best CV F1-score:  0.46687207655089813


In [37]:
xgb_params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_lambda": [1, 3, 5, 10]
}

In [38]:
xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),
    param_distributions=xgb_params,
    n_iter=25,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [39]:
xgb_search.fit(x_train_processed, y_train)

print("Best parameters found: ", xgb_search.best_params_)
print("Best CV F1-score: ", xgb_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best parameters found:  {'subsample': 0.7, 'reg_lambda': 3, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.2, 'gamma': 0.5, 'colsample_bytree': 1.0}
Best CV F1-score:  0.46674720745035786


In [47]:
cat_params = {
    "iterations": [100, 200, 300, 500],
    "depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7],
    "random_strength": [0.1, 0.5, 1, 2]
}

In [48]:
cat_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        verbose=0,
        random_seed=42,
        thread_count=1,
        allow_writing_files=False
    ),
    param_distributions=cat_params,
    n_iter=25,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [49]:
cat_search.fit(x_train_processed, y_train)

print("Best parameters found: ", cat_search.best_params_)
print("Best CV F1-score: ", cat_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best parameters found:  {'random_strength': 0.5, 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 500, 'depth': 8}
Best CV F1-score:  0.4549809208802678


In [50]:
best_models = {
    "Decision Tree": dt_search.best_estimator_,
    "XGBoost": xgb_search.best_estimator_,
    "CatBoost": cat_search.best_estimator_
}

In [52]:
tuning_results = []

In [54]:
for model_name, model in best_models.items():
    y_pred = model.predict(x_val_processed)

    tuning_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_val, y_pred, pos_label=1, zero_division=0),
        "F1-Score": f1_score(y_val, y_pred, pos_label=1, zero_division=0),
        "confusion_matrix": confusion_matrix(y_val, y_pred, labels=[0, 1]).tolist()
    })

In [55]:
tuning_df = pd.DataFrame(tuning_results)

In [56]:
tuning_df.to_csv(ROOT_DIR / "reports" / "tuning_results.csv", index=False)